# Hugging Face Fundamentals — Lesson 8: Model Cache

> Learning material for **Hugging Face Fundamentals**. Companion to the lesson script `08_Model_Cache.py` (same content, runnable without Jupyter).

**Task ID:** HF-008  |  **Folder:** `08_Model_Cache`


## Where do downloaded models live?

Every `from_pretrained` / `pipeline` download lands in a **cache** — a folder on your disk. The default is:

> `~/.cache/huggingface/hub`  (override with the `HF_HOME` env variable)

Inside, every model gets a `models--OWNER--NAME` folder. You never need to manage it by hand, but understanding it helps you free disk space and debug — and you *can* delete anything; it just re-downloads.

## Inspect the cache with the official API


In [ ]:
from huggingface_hub import scan_cache_dir

info = scan_cache_dir()
print("models cached :", len(info.repos))
print("total size    :", info.size_on_disk_str)


**List the models, biggest first:**

In [ ]:
for repo in sorted(info.repos, key=lambda r: r.size_on_disk, reverse=True):
    print(f"{repo.repo_id:<55} {repo.size_on_disk_str}")


## The layout inside one model folder

Each cached model keeps **two** copies of the data strategy: a `snapshots/` view you can load from, and `blobs/` holding the actual bytes (shared between revisions to save space — symlinks point to them). Revisions = different commits of the same model.


In [ ]:
import os
hub = os.path.expanduser("~/.cache/huggingface/hub")
first = sorted(os.listdir(hub))[0] if os.path.isdir(hub) else None
print("example cache entry:", first)
if first:
    print(os.listdir(os.path.join(hub, first)))


## Measure, delete, re-download

Disk space is precious — `delete_revisions` frees it. Loading the model again later simply downloads it anew:


In [ ]:
from huggingface_hub import delete_revisions, scan_cache_dir

info = scan_cache_dir()
target = next((r for r in info.repos if "tiny-distilbert" in r.repo_id), None)
if target:
    delete_revisions(
        repo_id=target.repo_id,
        revisions=[rev.commit_hash for rev in target.revisions],
    )
    print("deleted", target.repo_id, "-", target.size_on_disk_str)
else:
    print("no tiny-distilbert model in cache — nothing to delete")


## Offline mode doubles as a cache check

`HF_HUB_OFFLINE=1` blocks all network — a successful load proves the model is cached:


In [ ]:
os.environ["HF_HUB_OFFLINE"] = "1"
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    print("offline load OK -> cached")
finally:
    os.environ.pop("HF_HUB_OFFLINE", None)


## Try it yourself

1. Run `scan_cache_dir()` and note the total size.
2. Load a new small model, scan again — watch the size grow.
3. Delete one cached model, load it again — it re-downloads.

## Common pitfalls

- **Deleting is harmless** — but means a re-download later.
- **Windows without Developer Mode** — `huggingface_hub` warns it cannot use symlinks; caching still works, using a bit more disk.
- **`HF_HOME` vs `HF_HUB_CACHE`** — keep it simple: set `HF_HOME` and everything (`cache`, `datasets`, ...) moves together.

## Summary

- Cache default: `~/.cache/huggingface/hub`, model folders `models--ORG--NAME`.
- `scan_cache_dir` lists; `delete_revisions` frees space.
- `HF_HUB_OFFLINE=1` = cache-only mode; `HF_HOME` moves the cache.

This is the end of the module.  |  Extra reading: `../resources/reference_links.md`
